# 01 — Recto / verso folders and main-document sample

From `images/archive-original/`:

1. Split each photo pair: `o.jpg` → recto (text-bearing face), `m.jpg` → verso.
2. Write two flat PNG folders with **EXIF orientation baked into the pixels**.
   The PNGs carry no Orientation tag. A later user opens them upright and does
   not read EXIF.
3. On rectos, sample the **first** image of each metadata series (same
   `archief` + `fonds` + `signatuur` + date). Extra frames stay on disk but
   are not the main document.

Identity fields live on the verso row of `metadata.xlsx`; they are joined on
`volgnummer`.

In [ ]:
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from zipfile import ZipFile
from xml.etree import ElementTree as ET
import csv
import html
import random

from PIL import Image, ImageOps, ImageFile
try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **k: x

Image.MAX_IMAGE_PIXELS = None
ImageFile.LOAD_TRUNCATED_IMAGES = True

ROOT = Path("..").resolve()
SRC = ROOT / "images" / "archive-original"
META = ROOT / "images" / "metadata.xlsx"
RECTO_DIR = ROOT / "images" / "pages-recto"
VERSO_DIR = ROOT / "images" / "pages-verso"
MANIFEST = ROOT / "data" / "manifest.csv"
QC_HTML = ROOT / "outputs" / "stage1_pages_qc.html"

NS = {"m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
ORIENT_NAME = {
    1: "native", 2: "mirror-h", 3: "180", 4: "mirror-v",
    5: "transpose", 6: "90-cw", 7: "transverse", 8: "90-ccw",
}


def col_row(ref):
    col, i = 0, 0
    while i < len(ref) and ref[i].isalpha():
        col = col * 26 + (ord(ref[i].upper()) - 64)
        i += 1
    return col - 1, int(ref[i:])


def load_xlsx(path):
    with ZipFile(path) as z:
        strings = []
        shared = ET.fromstring(z.read("xl/sharedStrings.xml"))
        for si in shared:
            strings.append("".join(
                t.text or "" for t in si.iter(
                    "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}t")))
        root = ET.fromstring(z.read("xl/worksheets/sheet1.xml"))
        rows = []
        for row in root.find("m:sheetData", NS):
            cells = {}
            for c in row:
                ref = c.get("r")
                if not ref:
                    continue
                ci, _ = col_row(ref)
                t, v = c.get("t"), c.find("m:v", NS)
                val = (strings[int(v.text)] if t == "s" and v is not None
                       else (v.text if v is not None else ""))
                cells[ci] = val or ""
            if cells:
                mx = max(cells)
                rows.append([cells.get(i, "") for i in range(mx + 1)])
        return rows


def bake_png(src: Path, dst: Path):
    """Pixels in viewing orientation; PNG has no Orientation tag."""
    im = Image.open(src)
    orient = im.getexif().get(0x0112)  # Orientation
    raw_size = im.size
    im = ImageOps.exif_transpose(im)
    im = im.convert("RGB")
    dst.parent.mkdir(parents=True, exist_ok=True)
    im.save(dst, "PNG", compress_level=1)
    return {
        "exif_orientation": orient if orient is not None else "",
        "exif_name": ORIENT_NAME.get(orient, "missing"),
        "rotation_applied": int(orient not in (None, 1)),
        "width_raw": raw_size[0], "height_raw": raw_size[1],
        "width": im.size[0], "height": im.size[1],
    }

## Inventory, metadata join, main-document sample

In [ ]:
jpgs = sorted(p for p in SRC.rglob("*.jpg") if p.is_file())
assert jpgs, f"no jpgs in {SRC}"

photos = []
for p in jpgs:
    name = p.name.lower()
    if name.endswith("o.jpg"):
        side = "recto"
    elif name.endswith("m.jpg"):
        side = "verso"
    else:
        side = "other"
    photos.append({
        "original_path": str(p.relative_to(ROOT)),
        "src": p,
        "filename": p.name,
        "volgnummer": int(p.stem[:-1]),
        "side": side,
        "folder": p.parent.name,
    })

n_side = defaultdict(int)
for ph in photos:
    n_side[ph["side"]] += 1
print("files", dict(n_side))

header, *body = load_xlsx(META)
W = len(header)
raw = [dict(zip(header, (r + [""] * W)[:W])) for r in body]
meta_by_id = {}
for d in raw:
    fn = str(d.get("bestandsnaam", "")).lower()
    if fn.endswith("m.jpg"):
        meta_by_id[int(d["volgnummer"])] = {
            "archief": (d.get("archief") or "").strip(),
            "fonds": (d.get("fonds") or "").strip(),
            "signatuur": (d.get("signatuur") or "").strip(),
            "jaar": (d.get("jaar") or "").strip(),
            "maand": (d.get("maand") or "").strip(),
            "dag": (d.get("dag") or "").strip(),
            "extra_info": (d.get("extra info") or "").strip(),
        }

rectos = [ph for ph in photos if ph["side"] == "recto"]
groups = defaultdict(list)
for ph in rectos:
    m = meta_by_id.get(ph["volgnummer"], {})
    sig = m.get("signatuur") or ""
    if not sig:
        continue
    key = (m.get("archief", ""), m.get("fonds", ""), sig,
           m.get("jaar", ""), m.get("maand", ""), m.get("dag", ""))
    groups[key].append(ph["volgnummer"])

series_first = {}
for ids in groups.values():
    if len(ids) < 2:
        continue
    keep = min(ids)
    for i in ids:
        series_first[i] = keep

n_series = len({v for v in series_first.values()})
n_extra = sum(1 for i, k in series_first.items() if i != k)
print(f"series: {n_series}  extra frames dropped from gallery: {n_extra}")
print(f"main rectos: {len(rectos) - n_extra}")

## Bake orientation into PNG (recto + verso)

`ImageOps.exif_transpose` applies the tag and drops it. Saved PNGs are upright
RGB; there is no EXIF Orientation left to honour or ignore.

In [ ]:
RECTO_DIR.mkdir(parents=True, exist_ok=True)
VERSO_DIR.mkdir(parents=True, exist_ok=True)

jobs = []
for ph in photos:
    if ph["side"] == "recto":
        dst = RECTO_DIR / f"{ph['volgnummer']}o.png"
    elif ph["side"] == "verso":
        dst = VERSO_DIR / f"{ph['volgnummer']}m.png"
    else:
        continue
    ph["released"] = dst
    jobs.append(ph)


def _run(ph):
    info = bake_png(ph["src"], ph["released"])
    return ph["original_path"], info


baked = {}
with ThreadPoolExecutor(max_workers=4) as pool:
    futs = [pool.submit(_run, ph) for ph in jobs]
    for fut in tqdm(as_completed(futs), total=len(futs), desc="Bake PNG"):
        path, info = fut.result()
        baked[path] = info

n_rot = sum(1 for i in baked.values() if i["rotation_applied"])
print(f"wrote {len(baked)} PNGs  ({n_rot} had a non-native Orientation tag)")
print(f"recto → {RECTO_DIR}")
print(f"verso → {VERSO_DIR}")

## Manifest

In [ ]:
rows = []
for ph in photos:
    m = meta_by_id.get(ph["volgnummer"], {})
    info = baked.get(ph["original_path"], {})
    first = series_first.get(ph["volgnummer"])
    if ph["side"] == "verso":
        main, reason = 0, "verso"
    elif ph["side"] == "other":
        main, reason = 0, "unknown_suffix"
    elif first is not None and first != ph["volgnummer"]:
        main, reason = 0, f"series_of={first}o"
    elif first is not None:
        main, reason = 1, "series_first"
    else:
        main, reason = 1, "recto"
    rel = ""
    if "released" in ph:
        rel = str(ph["released"].relative_to(ROOT))
    rows.append({
        "volgnummer": ph["volgnummer"],
        "filename": ph["filename"],
        "side": ph["side"],
        "main_document": main,
        "reason": reason,
        "series_first": first if first is not None else "",
        "released_path": rel,
        "original_path": ph["original_path"],
        "folder": ph["folder"],
        "exif_orientation": info.get("exif_orientation", ""),
        "exif_name": info.get("exif_name", ""),
        "rotation_applied": info.get("rotation_applied", ""),
        "width": info.get("width", ""),
        "height": info.get("height", ""),
        **{k: m.get(k, "") for k in
           ("archief", "fonds", "signatuur", "jaar", "maand", "dag", "extra_info")},
    })

MANIFEST.parent.mkdir(parents=True, exist_ok=True)
fields = list(rows[0].keys())
with MANIFEST.open("w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=fields)
    w.writeheader()
    w.writerows(rows)

n_main = sum(1 for r in rows if r["main_document"] == 1)
print(f"manifest → {MANIFEST}  ({len(rows)} rows, {n_main} main rectos)")

## QC — sample pairs (upright) and every series keep/drop

In [ ]:
import base64, io


def thumb(path, long=240):
    im = Image.open(path).convert("RGB")
    w, h = im.size
    s = long / max(w, h)
    im = im.resize((max(1, round(w * s)), max(1, round(h * s))), Image.LANCZOS)
    buf = io.BytesIO()
    im.save(buf, "JPEG", quality=70)
    return base64.b64encode(buf.getvalue()).decode()


def img_tag(path):
    if not path.exists():
        return "<em>missing</em>"
    return f'<img src="data:image/jpeg;base64,{thumb(path)}">'


random.seed(0)
ids = sorted({ph["volgnummer"] for ph in photos})
sample_ids = sorted(random.sample(ids, 12))

pair_blocks = ["<h2>Random o / m pairs (baked, upright)</h2>"]
for pid in sample_ids:
    pair_blocks.append(
        f"<div class='row'><div class='cell'><div class='cap'>{pid}o recto</div>"
        f"{img_tag(RECTO_DIR / f'{pid}o.png')}</div>"
        f"<div class='cell'><div class='cap'>{pid}m verso</div>"
        f"{img_tag(VERSO_DIR / f'{pid}m.png')}</div></div>"
    )

series_blocks = ["<h2>Main document = first of series</h2>"]
by_first = defaultdict(list)
for i, keep in series_first.items():
    by_first[keep].append(i)
for keep, members in sorted(by_first.items(), key=lambda kv: -len(kv[1])):
    members = sorted(members)
    cells = []
    for pid in members:
        label = "KEEP" if pid == keep else "drop"
        cells.append(
            f'<div class="cell {label.lower()}"><div class="cap">{label} {pid}o</div>'
            f"{img_tag(RECTO_DIR / f'{pid}o.png')}</div>"
        )
    extra = next(r for r in rows if r["volgnummer"] == keep and r["side"] == "recto")
    title = html.escape(
        f"{extra['archief']} / {extra['fonds']} / {extra['signatuur']}  "
        f"{extra['jaar']}-{extra['maand']}-{extra['dag']}")
    series_blocks.append(f"<h3>{title}</h3><div class='row'>{''.join(cells)}</div>")

rotated = [r for r in rows if r["rotation_applied"] == 1 and r["side"] == "recto"]
rot_sample = rotated[:8]
rot_blocks = ["<h2>EXIF was applied (these would have been sideways as stored)</h2>",
              "<p>Left = raw JPEG pixels, ignoring the tag. Right = released PNG.</p>"]
for r in rot_sample:
    src = ROOT / r["original_path"]
    raw = Image.open(src).convert("RGB")  # no transpose — as stored
    w, h = raw.size
    s = 200 / max(w, h)
    raw = raw.resize((max(1, round(w * s)), max(1, round(h * s))), Image.LANCZOS)
    buf = io.BytesIO()
    raw.save(buf, "JPEG", quality=70)
    raw_b64 = base64.b64encode(buf.getvalue()).decode()
    rot_blocks.append(
        f"<div class='row'><div class='cell'><div class='cap'>raw {r['filename']} "
        f"({r['exif_name']})</div><img src='data:image/jpeg;base64,{raw_b64}'></div>"
        f"<div class='cell'><div class='cap'>PNG {r['volgnummer']}o</div>"
        f"{img_tag(RECTO_DIR / (str(r['volgnummer']) + 'o.png'))}</div></div>"
    )

QC_HTML.parent.mkdir(parents=True, exist_ok=True)
QC_HTML.write_text(
    "<!doctype html><meta charset=utf-8><title>01 pages QC</title>"
    "<style>body{font-family:system-ui;margin:24px;background:#111;color:#eee}"
    "h2{margin-top:32px}.row{display:flex;flex-wrap:wrap;gap:8px;margin:8px 0}"
    ".cell{width:200px}.cell img{width:200px;display:block;border-radius:4px}"
    ".cap{font:12px ui-monospace,monospace;margin-bottom:4px}"
    ".keep .cap{color:#3ddc84;font-weight:600}.drop .cap{color:#f0883e}</style>"
    f"<h1>Recto / verso + main document</h1>"
    f"<p>{len(baked)} PNGs · {n_rot} rotated from EXIF · {n_main} main rectos</p>"
    + "\n".join(rot_blocks + pair_blocks + series_blocks),
    encoding="utf-8",
)
print(f"QC → {QC_HTML}")